# NB01 — Dataset discovery, acquisition and provenance
Downloads every dataset used in the study, validates each file, and records URL, size, SHA-256 and date in the
run manifest. **No sample counts are assumed**: the inventory is built from the files actually downloaded.

| Dataset | Role | Mode |
|---|---|---|
| GSE4290, GSE50161 (GEO microarray) | tumour vs control (H2) | smoke + full |
| TCGA-GBM + GTEx via UCSC Xena "Toil" | tumour vs GTEx cortex (H2); TCGA survival cohort (H1 training) | phenotype in smoke, counts in full |
| TCGA-GBM survival / mutations (GDC hub on Xena) | survival, IDH status | smoke + full |
| CGGA mRNAseq_693 and _325 | H1 validation cohorts | full |
| GSE131928 | single-cell malignant-cell definition (M) | full |
| Ivy GAP RNA-seq | H3 (exploratory spatial) | full |
| SynGO bulk download | synaptic gene set S | smoke + full |
| Human Protein Atlas single-cell-type RNA | orthogonal check (NB12) | full |

**Endpoints are taken from public documentation patterns and were NOT tested from the authoring environment**
(its network is restricted). Every download tries the listed URLs and fails with instructions for a manual fallback.
Files already present are never re-downloaded.

In [ ]:
import os, sys
ROOT = os.environ.get("GBM_ROOT", "/content/drive/MyDrive/GBM_MultiCohort_Research")
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    print("Not in Colab; GBM_ROOT =", ROOT)
if not os.path.exists(os.path.join(ROOT, "00_Project_Config", "gbm_common.py")):
    raise FileNotFoundError("Run NB00 first (it writes gbm_common.py and config.json).")
sys.path.insert(0, os.path.join(ROOT, "00_Project_Config"))
from gbm_common import *
import zipfile, tarfile, json
print("SMOKE =", SMOKE, "| root =", ROOT)

Mounted at /content/drive
SMOKE = False | root = /content/drive/MyDrive/GBM_MultiCohort_Research


In [ ]:
RAW = P("raw")
URL = CFG["urls"]
inventory = []

def attempt(dataset, role, fname, urls, required, subdir="", min_bytes=1024):
    dest = RAW / subdir / fname if subdir else RAW / fname
    rec = {"dataset": dataset, "role": role, "file": str(dest.relative_to(RAW)), "required": required}
    try:
        download(urls, dest, min_bytes=min_bytes, key=f"{dataset}:{fname}")
        rec.update(status="ok", bytes=dest.stat().st_size, sha256=sha256(dest))
    except ActionableError as exc:
        rec.update(status="FAILED", bytes=0, sha256="", error=str(exc)[:600])
        if required:
            raise
        log(f"optional download failed: {dataset}/{fname}", "WARN")
    inventory.append(rec)
    return dest

In [ ]:
# 1) GEO microarray series matrices (small)
geo_paths = {}
for acc in CFG["geo_microarray"]:
    url = GEO_URL.format(prefix=acc[:-3], acc=acc)
    geo_paths[acc] = attempt(acc, "tumour vs control microarray", f"{acc}_series_matrix.txt.gz", [url], True, subdir="GEO")

def geo_platform(path):
    with gzip.open(path, "rt", errors="replace") as fh:
        for line in fh:
            if line.startswith("!Series_platform_id"):
                return line.split("\t")[1].strip().strip('"')
    return None

for acc, p in geo_paths.items():
    plat = geo_platform(p)
    exp = CFG["geo_microarray"][acc]
    require(plat == exp, f"{acc}: platform is {plat}, but the pipeline expects {exp}. "
                         "Probe mapping in NB03 is platform-specific; do not continue until this is resolved.")
    log(f"{acc}: platform {plat} confirmed")

2026-10-20T05:43:19Z [INFO] downloaded GSE4290_series_matrix.txt.gz: 21.9 MB from https://ftp.ncbi.nlm.nih.gov/geo/series/GSE4nnn/GSE4290/matrix/GSE4290_series_matrix.txt.gz
2026-10-20T05:43:20Z [INFO] downloaded GSE50161_series_matrix.txt.gz: 15.0 MB from https://ftp.ncbi.nlm.nih.gov/geo/series/GSE50nnn/GSE50161/matrix/GSE50161_series_matrix.txt.gz
2026-10-20T05:43:20Z [INFO] GSE4290: platform GPL570 confirmed
2026-10-20T05:43:20Z [INFO] GSE50161: platform GPL570 confirmed


In [9]:
# Fix inaccessible UCSC Xena GDC phenotype endpoint

URL["tcga_gdc_phenotype"] = [
    "https://gdc.xenahubs.net/download/TCGA-GBM.GDC_phenotype.tsv.gz",
    "https://gdc-hub.s3.us-east-1.amazonaws.com/download/TCGA-GBM.GDC_phenotype.tsv.gz",
]

print("Updated TCGA-GBM phenotype URLs:")
print(URL["tcga_gdc_phenotype"])

Updated TCGA-GBM phenotype URLs:
['https://gdc.xenahubs.net/download/TCGA-GBM.GDC_phenotype.tsv.gz', 'https://gdc-hub.s3.us-east-1.amazonaws.com/download/TCGA-GBM.GDC_phenotype.tsv.gz']


In [10]:
import requests

candidates = [
    "https://gdc.xenahubs.net/download/TCGA-GBM.GDC_phenotype.tsv.gz",
    "https://gdc-hub.s3.us-east-1.amazonaws.com/download/TCGA-GBM.GDC_phenotype.tsv.gz",
]

for url in candidates:
    try:
        response = requests.get(
            url,
            headers={"Range": "bytes=0-1023"},
            timeout=30,
            stream=True,
            allow_redirects=True,
        )

        print("Status:", response.status_code)
        print("URL:", response.url)
        print("Content-Type:", response.headers.get("Content-Type"))
        response.close()

    except requests.RequestException as e:
        print("Failed:", str(e))

Status: 403
URL: https://gdc-hub.s3.dualstack.us-east-1.amazonaws.com:443/download/TCGA-GBM.GDC_phenotype.tsv.gz
Content-Type: application/xml
Status: 403
URL: https://gdc-hub.s3.us-east-1.amazonaws.com/download/TCGA-GBM.GDC_phenotype.tsv.gz
Content-Type: application/xml


In [16]:
import os
import sys
import gzip
import json
import zipfile
import tarfile
import shutil
import pandas as pd
import requests
from pathlib import Path

ROOT = Path(os.environ.get(
    "GBM_ROOT",
    "/content/drive/MyDrive/GBM_MultiCohort_Research"
))

RAW = ROOT / "02_Raw_Datasets"
RAW.mkdir(parents=True, exist_ok=True)

inventory = []

def find_existing(fname, dataset=None, role=None):
    """Find an existing dataset file anywhere inside the raw-data folder."""

    # Exact filename match, including files in nested dataset folders.
    matches = [
        p for p in RAW.rglob("*")
        if p.is_file() and p.name.lower() == fname.lower()
    ]

    if matches:
        # Prefer the expected dataset subfolder when possible.
        if dataset:
            preferred = [
                p for p in matches
                if dataset.lower().replace("_", "") in
                str(p.parent).lower().replace("_", "")
            ]
            if preferred:
                return preferred[0]
        return matches[0]

    # Support the CGGA filenames visible in your screenshot.
    if dataset and dataset.upper().startswith("CGGA"):
        tag = "693" if "693" in dataset else "325"
        kind = (role or "").lower()

        for p in RAW.rglob("*"):
            if not p.is_file():
                continue

            name = p.name.lower()
            if (
                "cgga" in name
                and tag in name
                and (
                    ("clinical" in name and "clin" in kind)
                    or (
                        ("rsem-genes" in name or "rsem_genes" in name)
                        and ("expr" in kind or "expression" in kind)
                    )
                )
                and name.endswith(".zip")
            ):
                return p

    return None


def attempt(
    dataset, role, fname, urls, required,
    subdir="", min_bytes=1024
):
    """Reuse a local file where possible; otherwise download and validate."""

    dest = RAW / subdir / fname if subdir else RAW / fname
    existing = find_existing(fname, dataset, role)

    rec = {
        "dataset": dataset,
        "role": role,
        "file": "",
        "required": required,
    }

    try:
        if existing is not None:
            dest = existing
            size = dest.stat().st_size

            if size < min_bytes:
                raise ActionableError(
                    f"Existing file is too small: {dest} ({size} bytes)"
                )

            print(f"REUSED: {dest.relative_to(RAW)}")

        else:
            dest.parent.mkdir(parents=True, exist_ok=True)
            download(
                urls,
                dest,
                min_bytes=min_bytes,
                key=f"{dataset}:{fname}"
            )
            print(f"DOWNLOADED: {dest.relative_to(RAW)}")

        rec.update(
            status="ok",
            file=str(dest.relative_to(RAW)),
            bytes=dest.stat().st_size,
            sha256=sha256(dest),
        )

    except Exception as exc:
        # Do not silently ignore errors for required resources.
        rec.update(
            status="FAILED",
            bytes=0,
            sha256="",
            error=str(exc)[:600],
        )

        if required:
            inventory.append(rec)
            raise

        print(f"WARNING: {dataset}/{fname}: {exc}")

    inventory.append(rec)
    return dest

In [17]:
from google.colab import drive
drive.mount("/content/drive")

ROOT = Path(
    "/content/drive/MyDrive/GBM_MultiCohort_Research"
)

config_dir = ROOT / "00_Project_Config"
common_file = config_dir / "gbm_common.py"

if not common_file.exists():
    raise FileNotFoundError(
        f"Cannot find {common_file}. "
        "Check the Drive folder name and run NB00 first."
    )

sys.path.insert(0, str(config_dir))

from gbm_common import *

RAW = ROOT / "02_Raw_Datasets"
RAW.mkdir(parents=True, exist_ok=True)

URL = CFG["urls"]

print("Project root:", ROOT)
print("Raw data folder:", RAW)
print("Raw data folder exists:", RAW.exists())
print("Smoke mode:", SMOKE)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Project root: /content/drive/MyDrive/GBM_MultiCohort_Research
Raw data folder: /content/drive/MyDrive/GBM_MultiCohort_Research/02_Raw_Datasets
Raw data folder exists: True
Smoke mode: False


In [18]:
for p in sorted(RAW.glob("CGGA*.zip")):
    print(p.name, "|", p.stat().st_size, "bytes")

for tag in ["325", "693"]:
    for kind in ["expr", "clin"]:
        key = f"cgga{tag}_{kind}"
        fname = Path(URL[key][0].split("file=")[-1]).name

        found = find_existing(fname, f"CGGA_{tag}", kind)

        print(
            f"{key}:",
            found if found else f"NOT FOUND: {fname}"
        )

CGGA.mRNAseq_325.RSEM-genes.20200506.txt.zip | 14292049 bytes
CGGA.mRNAseq_325_clinical.20200506.txt.zip | 4546 bytes
CGGA.mRNAseq_693.RSEM-genes.20200506.txt.zip | 29738046 bytes
CGGA.mRNAseq_693_clinical.20200506.txt.zip | 8418 bytes
cgga325_expr: /content/drive/MyDrive/GBM_MultiCohort_Research/02_Raw_Datasets/CGGA.mRNAseq_325.RSEM-genes.20200506.txt.zip
cgga325_clin: /content/drive/MyDrive/GBM_MultiCohort_Research/02_Raw_Datasets/CGGA.mRNAseq_325_clinical.20200506.txt.zip
cgga693_expr: /content/drive/MyDrive/GBM_MultiCohort_Research/02_Raw_Datasets/CGGA.mRNAseq_693.RSEM-genes.20200506.txt.zip
cgga693_clin: /content/drive/MyDrive/GBM_MultiCohort_Research/02_Raw_Datasets/CGGA.mRNAseq_693_clinical.20200506.txt.zip


In [20]:
expected_dirs = [
    "CGGA", "GEO", "GSE131928", "HPA",
    "IvyGAP", "SynGO", "TCGA", "TOIL"
]

print("=== Dataset folders ===")
for name in expected_dirs:
    path = RAW / name
    print(f"{name:12} {'FOUND' if path.is_dir() else 'MISSING'}")

print("\n=== Existing CGGA archives ===")
for p in sorted(RAW.rglob("CGGA*.zip")):
    print(f"{p.relative_to(RAW)} | {p.stat().st_size:,} bytes")

print("\n=== Existing dataset file count ===")
files = [p for p in RAW.rglob("*") if p.is_file()]
print("Files:", len(files))

=== Dataset folders ===
CGGA         FOUND
GEO          FOUND
GSE131928    FOUND
HPA          FOUND
IvyGAP       FOUND
SynGO        FOUND
TCGA         FOUND
TOIL         FOUND

=== Existing CGGA archives ===
CGGA.mRNAseq_325.RSEM-genes.20200506.txt.zip | 14,292,049 bytes
CGGA.mRNAseq_325_clinical.20200506.txt.zip | 4,546 bytes
CGGA.mRNAseq_693.RSEM-genes.20200506.txt.zip | 29,738,046 bytes
CGGA.mRNAseq_693_clinical.20200506.txt.zip | 8,418 bytes

=== Existing dataset file count ===
Files: 19


In [24]:
# Replace NB01 section '# 3) Synaptic gene sets' through before '# 4) Full-mode only'
from pathlib import Path
import hashlib
import pandas as pd

syngo_dir = RAW / 'SynGO'
required_files = ['annotations.xlsx', 'genes.xlsx', 'ontologies.xlsx']
missing = [f for f in required_files if not (syngo_dir / f).is_file()]
if missing:
    raise FileNotFoundError(f'Upload these files into {syngo_dir}: {missing}')

required_columns = {
    'annotations.xlsx': {'hgnc_symbol', 'go_id', 'go_domain'},
    'genes.xlsx': {'hgnc_symbol', 'ensembl_id', 'entrez_id'},
    'ontologies.xlsx': {'id', 'domain', 'hgnc_symbol'},
}
frames = {}
for fname in required_files:
    path = syngo_dir / fname
    df = pd.read_excel(path, engine='openpyxl')
    absent = required_columns[fname] - set(df.columns)
    if absent:
        raise ValueError(f'{fname} missing columns: {sorted(absent)}')
    frames[fname] = df
    digest = hashlib.sha256(path.read_bytes()).hexdigest()
    inventory.append({
        'dataset': 'SynGO', 'role': fname.removesuffix('.xlsx'),
        'file': str(path.relative_to(RAW)), 'required': True,
        'status': 'ok', 'bytes': path.stat().st_size, 'sha256': digest,
    })
    print(f'VALIDATED {fname}: {len(df):,} rows')

syngo_genes = frames['genes.xlsx'].copy()
syngo_genes['hgnc_symbol'] = syngo_genes['hgnc_symbol'].astype('string').str.strip()
syngo_genes = syngo_genes.dropna(subset=['hgnc_symbol'])
syngo_genes = syngo_genes[syngo_genes['hgnc_symbol'].ne('')]
syngo_symbols = set(syngo_genes['hgnc_symbol'])

# Ontology gene lists are comma-separated. Preserve GO domain and ID.
terms = []
for _, row in frames['ontologies.xlsx'].iterrows():
    if pd.isna(row['id']) or pd.isna(row['hgnc_symbol']):
        continue
    for symbol in str(row['hgnc_symbol']).split(','):
        symbol = symbol.strip()
        if symbol:
            terms.append({'go_id': str(row['id']).strip(),
                          'domain': str(row['domain']).strip(),
                          'hgnc_symbol': symbol})
term_membership = pd.DataFrame(terms).drop_duplicates()

# Save derived analysis files outside raw inputs.
processed = ROOT / '03_Processed_Datasets' / 'SynGO'
processed.mkdir(parents=True, exist_ok=True)
syngo_genes.to_csv(processed / 'syngo_genes.csv', index=False)
term_membership.to_csv(processed / 'syngo_term_membership.csv', index=False)

print('SynGO unique gene symbols:', len(syngo_symbols))
print('Ontology membership rows:', len(term_membership))
print('Ontology genes absent from genes.xlsx:',
      len(set(term_membership['hgnc_symbol']) - syngo_symbols))
print('Processed files saved to:', processed)

VALIDATED annotations.xlsx: 5,076 rows
VALIDATED genes.xlsx: 1,789 rows
VALIDATED ontologies.xlsx: 303 rows
SynGO unique gene symbols: 1789
Ontology membership rows: 14050
Ontology genes absent from genes.xlsx: 0
Processed files saved to: /content/drive/MyDrive/GBM_MultiCohort_Research/03_Processed_Datasets/SynGO


In [25]:
# 4) Full-mode only: CGGA, single-cell, Ivy GAP, HPA
def unzip_first(zpath, outdir):
    outdir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zpath) as z:
        z.extractall(outdir)
    return sorted(p for p in outdir.rglob("*") if p.is_file())

if not SMOKE:
    for tag in ["693", "325"]:
        for kind in ["expr", "clin"]:
            key = f"cgga{tag}_{kind}"
            fn = Path(URL[key][0].split("file=")[-1]).name
            zp = attempt(f"CGGA_{tag}", kind, fn, URL[key], False, subdir="CGGA")
            if zp.exists():
                unzip_first(zp, RAW / "CGGA" / f"extracted_{tag}")
    attempt("GSE131928", "single-cell raw tar", "GSE131928_RAW.tar", URL["gse131928_raw"], False, subdir="GSE131928", min_bytes=1_000_000)
    tarp = RAW / "GSE131928" / "GSE131928_RAW.tar"
    if tarp.exists():
        with tarfile.open(tarp) as t:
            t.extractall(RAW / "GSE131928" / "extracted")
    ivy = attempt("IvyGAP", "RNA-seq anatomic structures", "ivygap_gene_expression_matrix.zip", URL["ivy_expression"], False, subdir="IvyGAP")
    if ivy.exists():
        try:
            unzip_first(ivy, RAW / "IvyGAP" / "extracted")
        except zipfile.BadZipFile:
            log("Ivy GAP download is not a zip; see troubleshooting in the guide.", "WARN")
    attempt("HPA", "single-cell-type RNA", "rna_single_cell_type.tsv.zip", URL["hpa_sc"], False, subdir="HPA")

REUSED: CGGA.mRNAseq_693.RSEM-genes.20200506.txt.zip
REUSED: CGGA.mRNAseq_693_clinical.20200506.txt.zip
REUSED: CGGA.mRNAseq_325.RSEM-genes.20200506.txt.zip
REUSED: CGGA.mRNAseq_325_clinical.20200506.txt.zip
REUSED: GSE131928/GSE131928_RAW.tar


/tmp/ipykernel_686/3568899950.py:20: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  t.extractall(RAW / "GSE131928" / "extracted")


REUSED: IvyGAP/ivygap_gene_expression_matrix.zip
REUSED: HPA/rna_single_cell_type.tsv.zip


In [26]:
# Format validation: parse the GEO files and report what is actually inside (no assumed counts).
for acc, p in geo_paths.items():
    m, e = read_geo_series_matrix(p)
    print(f"{acc}: {m.shape[0]} samples x {e.shape[0]} probes; metadata columns: {list(m.columns)[:12]}")
    manifest_update("acquisition", f"{acc}_parsed", {"n_samples": int(m.shape[0]), "n_probes": int(e.shape[0])})
print("\nNB01 complete. Next: NB02 (metadata harmonisation and QC).")

GSE4290: 180 samples x 54613 probes; metadata columns: ['title', 'geo_accession', 'status', 'submission_date', 'last_update_date', 'type', 'channel_count', 'source_name_ch1', 'organism_ch1', 'molecule_ch1', 'extract_protocol_ch1', 'label_ch1']
GSE50161: 130 samples x 21050 probes; metadata columns: ['title', 'geo_accession', 'status', 'submission_date', 'last_update_date', 'type', 'channel_count', 'source_name_ch1', 'organism_ch1', 'molecule_ch1', 'extract_protocol_ch1', 'label_ch1']

NB01 complete. Next: NB02 (metadata harmonisation and QC).
